# Transfermarkt Data Inspection

Phase 1 — Data Inspection, part 2. This notebook asks whether Transfermarkt's player records can supply prices that link to StatsBomb players.

Headline results:
- Prices exist as a timeline: player_valuations.csv (656k dated records, 2000–2026).
- Bio data fills StatsBomb's gap: birth dates, height, foot, position.
- The two datasets share no ID numbers, so players must be linked by name plus extra details (club, position, birth date).
- 937 names belong to 2+ players: name-only linking would attach wrong prices.

## Cell 1 — Setup and file list

Points at the Transfermarkt folder (plus the StatsBomb folder, needed for the league comparison in Cell 9) and lists every file with its size, so the three giant files are known before anything opens them.

In [1]:
import os
import pandas as pd
import numpy as np

TM_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'transfermarkt'))
SB_DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'statsbomb', 'data'))

print('tm exists:', os.path.isdir(TM_DIR))
for f in sorted(os.listdir(TM_DIR)):
    fp = os.path.join(TM_DIR, f)
    if os.path.isfile(fp):
        print(f'{f}: {os.path.getsize(fp) / 1024 ** 2:.1f} MB')


tm exists: True
appearances.csv: 189.2 MB
club_games.csv: 14.1 MB
clubs.csv: 0.2 MB
competitions.csv: 0.0 MB
countries.csv: 0.0 MB
game_events.csv: 170.8 MB
game_lineups.csv: 396.7 MB
games.csv: 27.6 MB
national_teams.csv: 0.0 MB
player_valuations.csv: 37.8 MB
players.csv: 18.7 MB
transfers.csv: 17.3 MB


## Cell 2 — Players table (registry and bio data)

One row per player (50k). Beyond the current-price snapshot, this table holds the details that tell same-named players apart (club, position, birth date) plus the bio fields StatsBomb lacks.

In [2]:
df_players = pd.read_csv(os.path.join(TM_DIR, 'players.csv'))
print(df_players.shape)
print(list(df_players.columns))
print()
print('unique player_id:', df_players['player_id'].nunique())
print('dup player_id rows:', df_players['player_id'].duplicated().sum())
df_players.head()


(50149, 26)
['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'international_caps', 'international_goals', 'current_national_team_id', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'market_value_in_eur', 'highest_market_value_in_eur']

unique player_id: 50149
dup player_id rows: 0


,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,agent_name,image_url,international_caps,international_goals,current_national_team_id,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,ASBW Sport Marketing,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/miroslav-klose...,IT1,Società Sportiva Lazio S.p.A.,1000000.0,30000000.0
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000.0,8000000.0
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,CSKA-AS-23 Ltd.,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/dimitar-berbat...,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000.0,34500000.0
3,77,NaN,Lúcio,Lúcio,2012,506,lucio,Brazil,Brasília,Brazil,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/lucio/profil/s...,IT1,Juventus FC,200000.0,24500000.0
4,80,Tom,Starke,Tom Starke,2017,27,tom-starke,East Germany (GDR),Freital,Germany,...,IFM,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/tom-starke/pro...,L1,Bayern Munich,100000.0,3000000.0


## Cell 3 — Price history (the target variable)

Each row is one dated price observation. A timeline per player means attaching the price as of a given season instead of today's price to old performances. Prices arrive sparsely (around 16 per player), so the join uses the latest price before season end.

In [3]:
df_val = pd.read_csv(os.path.join(TM_DIR, 'player_valuations.csv'), parse_dates=['date'])
print(df_val.shape)
print('range:', df_val['date'].min().date(), '->', df_val['date'].max().date())
print('unique valued players:', df_val['player_id'].nunique())
print()
print('valuations per player:')
print(df_val.groupby('player_id').size().describe())
df_val.head()


(656301, 6)
range: 2000-01-20 -> 2026-06-12
unique valued players: 41528

valuations per player:
count    41528.000000
mean        15.803819
std          9.823933
min          1.000000
25%          8.000000
50%         15.000000
75%         23.000000
max         57.000000
dtype: float64


,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id
0,405973,2000-01-20,150000,Unknown,3057,BE1
1,342216,2001-07-20,100000,Unknown,1241,SC1
2,3132,2003-12-09,400000,Dynamo Kyiv,126,TR1
3,6893,2003-12-15,900000,Galatasaray,984,GB1
4,10,2004-10-04,7000000,SV Werder Bremen,398,IT1


## Cell 4 — Transfers table (fees)

Transfer fees plus each player's price at transfer time. Empty fees mean loans and free transfers — normal, not missing data.

In [4]:
df_transfers = pd.read_csv(os.path.join(TM_DIR, 'transfers.csv'), parse_dates=['transfer_date'])
print(df_transfers.shape)
print(list(df_transfers.columns))
print()
print('null transfer_fee:', df_transfers['transfer_fee'].isna().sum())
print('null market_value_in_eur:', df_transfers['market_value_in_eur'].isna().sum())
df_transfers.head()


(175165, 10)
['player_id', 'transfer_date', 'transfer_season', 'from_club_id', 'to_club_id', 'from_club_name', 'to_club_name', 'transfer_fee', 'market_value_in_eur', 'player_name']

null transfer_fee: 61526
null market_value_in_eur: 68160


,player_id,transfer_date,transfer_season,from_club_id,to_club_id,from_club_name,to_club_name,transfer_fee,market_value_in_eur,player_name
0,467994,2030-06-30,25/26,5621,749,Reggiana,FC Empoli,0.0,700000.0,Luca Belardinelli
1,645842,2028-02-02,27/28,6505,19684,Gimcheon Sangmu,Jeju SK,0.0,150000.0,Chan-gi An
2,677470,2028-02-02,27/28,6505,3535,Gimcheon Sangmu,Ulsan HD,0.0,400000.0,Yool Heo
3,709155,2028-02-02,27/28,6505,19684,Gimcheon Sangmu,Jeju SK,0.0,325000.0,Jae-hyeok Oh
4,728133,2028-02-02,27/28,6505,35759,Gimcheon Sangmu,Bucheon FC,0.0,175000.0,Sang-hyeok Lee


## Cell 5 — Supporting tables (quick look only)

Column lists for games, appearances, lineups, events, clubs, competitions, and club-games via small samples — no full loads. appearances.csv (1.9M rows of per-game goals/assists/minutes) is saved as a future cross-check for StatsBomb-based totals.

In [5]:
for f in ['games.csv', 'appearances.csv', 'game_lineups.csv', 'game_events.csv', 'clubs.csv', 'competitions.csv', 'club_games.csv']:
    peek = pd.read_csv(os.path.join(TM_DIR, f), nrows=2)
    print(f'{f}: {len(peek.columns)} cols')
    print('  ', list(peek.columns))

df_games = pd.read_csv(os.path.join(TM_DIR, 'games.csv'), usecols=['game_id', 'date'])
df_games['date'] = pd.to_datetime(df_games['date'])
print()
print('games:', len(df_games), '| range:', df_games['date'].min().date(), '->', df_games['date'].max().date())


games.csv: 23 cols
   ['game_id', 'competition_id', 'season', 'round', 'date', 'home_club_id', 'away_club_id', 'home_club_goals', 'away_club_goals', 'home_club_position', 'away_club_position', 'home_club_manager_name', 'away_club_manager_name', 'stadium', 'attendance', 'referee', 'url', 'home_club_formation', 'away_club_formation', 'home_club_name', 'away_club_name', 'aggregate', 'competition_type']
appearances.csv: 13 cols
   ['appearance_id', 'game_id', 'player_id', 'player_club_id', 'player_current_club_id', 'date', 'player_name', 'competition_id', 'yellow_cards', 'red_cards', 'goals', 'assists', 'minutes_played']
game_lineups.csv: 10 cols
   ['game_lineups_id', 'date', 'game_id', 'player_id', 'club_id', 'player_name', 'type', 'position', 'number', 'team_captain']
game_events.csv: 11 cols
   ['game_event_id', 'date', 'game_id', 'minute', 'type', 'club_id', 'club_name', 'player_id', 'description', 'player_in_id', 'player_assist_id']
clubs.csv: 17 cols
   ['club_id', 'club_code', 'nam

## Cell 6 — Gaps, duplicates, and text encoding

Duplicate-ID checks, empty price counts (snapshot vs timeline), and a scan for broken characters in names. Non-English names (Muñoz, Peña) are where text encoding breaks and name-matching later fails, so this check comes before any matching. A clean scan clears the file for joining.

In [6]:
print('players dup player_id:', df_players['player_id'].duplicated().sum())
print('players null market_value_in_eur:', df_players['market_value_in_eur'].isna().sum())
print('valuations null market_value_in_eur:', df_val['market_value_in_eur'].isna().sum())
print()
for col in ['name', 'first_name', 'last_name']:
    bad = df_players[col].str.contains('\ufffd', na=False).sum()
    print(col, 'replacement-char rows:', bad)
print()
miss = df_players.isna().sum()
print('players cols with missing:', miss[miss > 0].to_dict())


players dup player_id: 0
players null market_value_in_eur: 8621
valuations null market_value_in_eur: 0

name replacement-char rows: 0
first_name replacement-char rows: 0
last_name replacement-char rows: 0

players cols with missing: {'first_name': 3164, 'country_of_birth': 5885, 'city_of_birth': 5608, 'country_of_citizenship': 269, 'date_of_birth': 49, 'sub_position': 586, 'foot': 5893, 'height_in_cm': 4365, 'contract_expiration_date': 18569, 'agent_name': 23296, 'international_caps': 30793, 'international_goals': 30793, 'current_national_team_id': 46889, 'current_club_domestic_competition_id': 2986, 'current_club_name': 2986, 'market_value_in_eur': 8621, 'highest_market_value_in_eur': 8621}


## Cell 7 — Do the ID numbers match StatsBomb?

Direct check, not assumption: Transfermarkt has no StatsBomb ID column, and its competition codes are letters (ES1, GB1) while StatsBomb's are numbers. Conclusion: no shared key exists — players must be linked by name plus details.

In [7]:
print('statsbomb key in TM players cols:',
      [c for c in df_players.columns if 'statsbomb' in c.lower()] or 'NONE')
df_tm_comp = pd.read_csv(os.path.join(TM_DIR, 'competitions.csv'), usecols=['competition_id', 'name'])
print('TM competition_id sample:', df_tm_comp['competition_id'].head(3).tolist())
print('TM player_id sample (Messi / Haaland):',
      df_players.loc[df_players['name'].isin(['Lionel Messi', 'Erling Haaland']), ['player_id', 'name']].to_dict('records'))


statsbomb key in TM players cols: NONE
TM competition_id sample: ['A1', 'AFAC', 'AFCN']
TM player_id sample (Messi / Haaland): [{'player_id': 28003, 'name': 'Lionel Messi'}, {'player_id': 418560, 'name': 'Erling Haaland'}]


## Cell 8 — Shared names (why names alone are dangerous)

Counts names belonging to 2+ players (Ronaldo and others) to show why joining on name alone would silently attach wrong prices. The Ronaldo rows show the columns that resolve them: club, position, birth date.

In [8]:
dup_names = df_players['name'].value_counts()
print('names shared by 2+ players:', (dup_names > 1).sum())
print()
print(dup_names.head(10))
print()
print(df_players.loc[df_players['name'] == 'Ronaldo',
      ['player_id', 'date_of_birth', 'position', 'current_club_name', 'market_value_in_eur']].head(10).to_string())


names shared by 2+ players: 937

name
Paulinho      14
João Pedro    13
Vitinho       11
Eduardo       10
Bruninho      10
Guilherme      9
Pedrinho       8
Danilo         8
Rafinha        8
Serginho       8
Name: count, dtype: int64

       player_id        date_of_birth    position         current_club_name  market_value_in_eur
10438     146660  1990-04-08 00:00:00    Midfield                 FC Empoli             400000.0
23215     387925  1996-08-22 00:00:00  Goalkeeper       Esporte Clube Bahia            2000000.0
24235     405768  1996-10-23 00:00:00    Midfield  Sport Club Internacional            2000000.0
39200     866670  2000-12-07 00:00:00      Attack                 FK Rostov            3500000.0


## Cell 9 — League overlap

Reads StatsBomb's 80-row competitions list and matches it against all Transfermarkt competitions (cups and internationals sit under other type labels, so league-only filtering would miss them). Name matching ignores punctuation and case, and every pair prints for review. Matched leagues mark where a performance-plus-price join is possible; the rest (notably all women's leagues) are coverage gaps.

In [9]:
import re
sb_comp = pd.read_json(os.path.join(SB_DATA_DIR, 'competitions.json'))

def norm(s):
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

tm_all = pd.read_csv(os.path.join(TM_DIR, 'competitions.csv'))
tm_all['key'] = tm_all['name'].apply(norm)
sb_comp['key'] = sb_comp['competition_name'].apply(norm)

tm_keys = tm_all[['competition_id', 'name', 'key']].drop_duplicates()
rows = []
for _, r in sb_comp.drop_duplicates('key').iterrows():
    hits = tm_keys[tm_keys['key'].apply(lambda t: r['key'] in t or t in r['key'])]
    rows.append((r['competition_name'], hits['competition_id'].tolist()))
m = pd.DataFrame(rows, columns=['sb_league', 'tm_ids'])
print('SB leagues:', m.shape[0], '| with TM match:', (m['tm_ids'].str.len() > 0).sum())
print()
print(m.to_string())


SB leagues: 24 | with TM match: 18

                  sb_league           tm_ids
0             1. Bundesliga         [A1, L1]
1    African Cup of Nations               []
2          Champions League        [CL, CLQ]
3              Copa America           [COPA]
4              Copa del Rey            [CDR]
5   FA Women's Super League             [C1]
6        FIFA U20 World Cup           [FIWC]
7            FIFA World Cup           [FIWC]
8         Frauen Bundesliga         [A1, L1]
9       Indian Super league             [C1]
10                  La Liga            [ES1]
11                   Liga F               []
12         Liga Profesional               []
13                  Ligue 1            [FR1]
14      Major League Soccer           [MLS1]
15    North American League               []
16                     NWSL               []
17           Premier League            [GB1]
18                  Serie A      [BRA1, IT1]
19            Serie A Women            [IT1]
20                U

## Next steps

1. Confirm the overlap table from Cell 9.
2. Prototype the name-cleaning plus club/position/birth-date join (notebook 03).
3. Attach prices-as-of-season-end to player-season rows.
4. Design the Postgres database from the two settled sources.